### Агрегация в двоичном дереве

В данном испытании мы будем использовать двоичное дерево, и выполнять агрегацию данных.

#### src/solution.py

**Реализуйте следующие методы в классе `solution.Node`:**

- `__len__()` — возвращает количество узлов в дереве (используется в `len()`).
- `__repr__()` — возвращает строковое представление дерева (используется для отображения в `REPL`).
- `total()` — возвращает сумму всех ключей дерева.
- `minimum()` — возвращает минимальный ключ дерева.
- `maximum()` — возвращает максимальный ключ дерева.
- `to_list()` — возвращает плоский список, содержащий все ключи.
- `every(fn)` — проверяет, удовлетворяют ли все ключи дерева условию, заданному в передаваемой функции.
- `some(fn)` — проверяет, удовлетворяет ли какой-либо ключ дерева условию, заданному в передаваемой функции.

При обходе дерева нужно использовать порядок слева-направо. То есть вначале обрабатываем ключ узла, затем ключ левого ребёнка, после чего ключ правого ребёнка.

```python
from solution import Node
tree = Node(
    9,
    Node(
        4,
        Node(8),
        Node(
            6,
            Node(3),
            Node(7),
        ),
    ),
    Node(
        17,
        right=Node(
            22,
            Node(20),
        ),
    ),
)
len(tree)  # 9
tree.total()  # 96
tree.to_list()  # [9, 4, 8, 6, 3, 7, 17, 22, 20]
tree.every(lambda key: key <= 22)  # True
tree.some(lambda key: key > 22)  # False
tree.minimum()  # 3
tree.maximum()  # 22
tree2 = Node(3, Node(1), Node(2))
tree2  # выводится repr(tree2)
# Node(3, Node(1, None, None), Node(2, None, None))
```

##### Подсказки

- Для реализации каждого из методов потребуется выполнить обход всех узлов дерева.
---


#### Разбор решения

##### 1. Алгоритм обхода дерева Pre-order (узел → слева → справа)

In [1]:
class Node:
    def __init__(self, key=None, left=None, right=None):
        self.key = key
        self.left = left
        self.right = right

    def insert(self, key):
        if self.key is None:
            self.key = key
            return
        if key == self.key:
            return
        if key < self.key:
            if not self.left:
                self.left = self.__class__()
            target = self.left
        else:
            if not self.right:
                self.right = self.__class__()
            target = self.right
        target.insert(key)
        
         
    def pre_order(self, result = None):
        if result is None:
            result = []
        if self.key is not None:
            result.append(self.key)
        if self.left:
            self.left.pre_order(result)
        if self.right:
            self.right.pre_order(result)
        return result
                    
tree = Node(
    9,
    Node(
        4,
        Node(2),
        Node(
            6,
            Node(3),
            Node(7),
        ),
    ),
    Node(
        17,
        right=Node(
            22,
            Node(20),
        ),
    ),
)

result = tree.pre_order()
print(result)

[9, 4, 2, 6, 3, 7, 17, 22, 20]


##### 2. Метод `every(fn)`
**проверяет, удовлетворяют ли все ключи дерева условию, заданному в передаваемой функции.**

In [ ]:
def every(self, predicate):
    def check(node):
        if node is None:
            return True
        if not predicate(node.key): # если ключ дерева не удовлетворяет условию в передаваемой функции
            return False
        return check(node.left) and check(node.right)
    return check(self)

- `check(node)` - вспомогательная рекурсивная функция

**Как работают действия этой функции**

**Передача предиката.** В метод `every` передаётся функция `predicate`, которая принимает один аргумент `(ключ узла)` и возвращает `True` или `False`. **Например**: `lambda x: x > 0` (все ключи больше нуля).

**Рекурсивный обход.**  Функция `check` рекурсивно обходит всё дерево:
- Если узел `None` (пустой), считаем, что в этом поддереве условие выполнено — возвращаем `True`. Это базовый случай рекурсии.

    - Здесь `check` может быть вызвана с `None`, потому что мы явно передаём туда детей: `check(node.left)`. 
    - Если у узла нет левого ребёнка, `node.left` — это `None`, и функция должна корректно обработать такой вызов.
    - Почему возвращаем `True`? Это «нейтральный» результат для операции «все элементы удовлетворяют условию». Логика такая:

        - «Все ключи в пустом поддереве удовлетворяют условию» — это истина по умолчанию (в логике это называется `vacuous truth`: 
        - если элементов нет, то и нарушить условие некому).
    - Если бы мы вернули `False`, то любое дерево с пустым поддеревом сразу считалось бы «не удовлетворяющим условию», даже если все реальные ключи подходят.
- Проверяем условие для ключа текущего узла: если `predicate(node.key)` — `False`, сразу возвращаем `False`  (оптимизация: дальше можно не проверять).
- Если условие для текущего узла выполнено, рекурсивно проверяем левое и правое поддеревья.

**Логическое И**. Результат проверки — это логическое `И` между проверкой текущего узла и проверкой обоих поддеревьев: `check(node.left) and check(node.right)`. Если хотя бы в одном месте условие нарушено, результат будет `False`.

**Возврат результата**. Внешний метод `every` запускает проверку с корня `(self)` и возвращает итоговый `True/False`.

---